
# Write a simulated photon stream, read it back

A TTTR container is a record stream plus a header. This example builds a photon
stream from a simulation -- no instrument file needed -- writes it as a
PicoQuant PTU (HydraHarp T3 and PicoHarp T3), a HydraHarp HT3 and a Becker & Hickl SPC-130 file
with :meth:`tttrlib.TTTR.write`, reads each back, and checks that every macro
time, micro time and routing channel survives. The record encoders insert the
overflow records the formats need (a 10-bit macro-time field wraps every 1024
ticks in HHT3v2, a 12-bit one in SPC-130), and the readers undo them; the
photons themselves are exact.

The same decoders are compared photon for photon with [phconvert](https://github.com/Photon-HDF5/phconvert) and [ptufile](https://github.com/cgohlke/ptufile) in the test suite and the benchmark
harness (``benchmarks/check_reading.py``): identical, and 5-25x faster.


In [ ]:
import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import tttrlib

## A simulated photon stream
Two detection channels, a Poisson background and a handful of bright
"bursts": inter-photon times are exponential, micro times follow a 2.5 ns
decay through a Gaussian IRF, and the routing channel is drawn per photon.



In [ ]:
rng = np.random.default_rng(7)
MACRO_RES = 25e-9            # 40 MHz sync -> 25 ns macro-time tick
MICRO_RES = 16e-12           # 16 ps micro-time bins
N_MICRO = 4096
T_TOTAL = 1.0                # seconds

rate = np.full(int(T_TOTAL / 1e-3), 3e3)              # 3 kHz background per ms
for c in rng.uniform(0.05, 0.95, 12):                  # 12 bursts of 3 ms at 60 kHz
    i = int(c / 1e-3)
    rate[i:i + 3] = 6e4
counts = rng.poisson(rate * 1e-3)
t = np.concatenate([np.sort(rng.uniform(i * 1e-3, (i + 1) * 1e-3, n))
                    for i, n in enumerate(counts)])
macro = (t / MACRO_RES).astype(np.uint64)
n = macro.size

tau_ticks = 2.5e-9 / MICRO_RES
irf_pos, irf_sig = 1.0e-9 / MICRO_RES, 0.12e-9 / MICRO_RES
micro = (rng.normal(irf_pos, irf_sig, n) + rng.exponential(tau_ticks, n))
micro = np.clip(np.round(micro), 0, N_MICRO - 1).astype(np.uint16)
channels = rng.choice([0, 2], size=n, p=[0.6, 0.4]).astype(np.int8)
print(f"{n} photons over {T_TOTAL} s, {counts.max()} in the brightest ms")

## A container from arrays
``append_events`` takes the four record columns; the header carries the two
resolutions and the number of micro-time channels the encoders need.



In [ ]:
data = tttrlib.TTTR()
data.append_events(macro, micro, channels, np.zeros(n, np.int8))
data.header.set_macro_time_resolution(MACRO_RES)
data.header.set_micro_time_resolution(MICRO_RES)
data.header.set_number_of_micro_time_channels(N_MICRO)


def round_trip(container, record_type, suffix, reader_hint):
    """Write with (container, record type), read back, compare every record."""
    fd, path = tempfile.mkstemp(suffix=suffix)
    os.close(fd)
    try:
        d = tttrlib.TTTR()
        d.append_events(macro, micro, channels, np.zeros(n, np.int8))
        d.header.set_macro_time_resolution(MACRO_RES)
        d.header.set_micro_time_resolution(MICRO_RES)
        d.header.set_number_of_micro_time_channels(N_MICRO)
        d.header.tttr_container_type = container
        d.header.tttr_record_type = record_type
        assert d.write(path)
        size = os.path.getsize(path)
        back = tttrlib.TTTR(path, reader_hint)
        ok = (np.array_equal(np.asarray(back.macro_times), macro)
              and np.array_equal(np.asarray(back.micro_times), micro)
              and np.array_equal(np.asarray(back.routing_channels), channels))
        return ok, size, back
    finally:
        if os.path.exists(path):
            os.remove(path)


# container / record type codes: PTU + HHT3v2, PTU + PicoHarp T3, HT3 + HHT3v2, SPC-130
results = {}
for name, container, record, suffix, hint in [
    ("PTU (HydraHarp T3)", 0, 4, ".ptu", "PTU"),
    ("PTU (PicoHarp T3)", 0, 5, ".ptu", "PTU"),
    ("HT3 (HydraHarp T3)", 1, 4, ".ht3", "HT3"),
    ("SPC-130 (Becker & Hickl)", 2, 0, ".spc", "SPC-130"),
]:
    ok, size, back = round_trip(container, record, suffix, hint)
    results[name] = (ok, size, back)
    print(f"{name:26s} {size / 1e3:8.1f} kB  {'identical' if ok else 'DIFFERS'}  "
          f"({len(back.macro_times)} photons read back)")

## What came back
The three files hold the same photons; the micro-time histogram and the
intensity trace of the SPC-130 read-back are those of the simulation.



In [ ]:
_, _, back = results["SPC-130 (Becker & Hickl)"]
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.4))
h = back.get_microtime_histogram(16)[0]
a.semilogy(np.arange(h.size) * MICRO_RES * 16 * 1e9, np.maximum(h, 0.5))
a.set_xlabel("micro time (ns)")
a.set_ylabel("photons")
a.set_title("decay through the IRF, read back from SPC-130")
trace = back.get_intensity_trace(1e-3)
b.plot(np.arange(len(trace)) * 1e-3, trace)
b.set_xlabel("time (s)")
b.set_ylabel("photons / ms")
b.set_title("intensity trace: 12 bursts on a 3 kHz background")
plt.tight_layout()
plt.show()

## Notes
* The overflow bookkeeping differs per format (HHT3v2 counts up to 1023
  overflows in one record; PicoHarp T3 wraps a 16-bit sync counter with one
  record per overflow; SPC-130 wraps a 12-bit macro time), which is why the
  files differ in size while the photons are identical.
* PicoHarp T3 keeps its markers on the special channel 15 with the marker
  bits in the micro time; a photon with micro time 0 stays a photon (the
  PicoQuant convention, identical to ptufile's decoding).
* ``TTTR.write`` fills the mandatory header tags a container needs
  (record type, bits per record, resolutions, record count) from the header
  given, and for PTU patches ``TTResult_NumberOfRecords`` to the number of
  records actually written -- events plus overflow records -- so strict readers
  see the whole stream.

